## GRU Language Model

In [ ]:
from datasets import load_dataset
import re
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import trange
from RNNUtils import plot_errors_accuracy
import math

### Download and prepare the Wikitext dataset from Huggingface

In [ ]:
# Load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

In [ ]:
tokenizer = get_tokenizer('basic_english')
tokenize_data = lambda example, tokenizer: {'tokens': tokenizer(example['text'])}  
tokenized_dataset = dataset.map(tokenize_data, remove_columns=['text'], fn_kwargs={'tokenizer': tokenizer})

In [ ]:
vocab = build_vocab_from_iterator(tokenized_dataset['train']['tokens'], min_freq=3) 

In [ ]:
vocab.insert_token('<unk>', 0)           
vocab.insert_token('<eos>', 1)            
vocab.set_default_index(vocab['<unk>'])   

In [ ]:
vocab_len = len(vocab)
print(vocab_len)                         
print(vocab.get_itos()[:10]) 

In [ ]:
def prepare_tokenized(dataset, vocab):
    data = []                                                   
    for item in dataset:
        if item['tokens']:                                      
            tokens = item['tokens'].append('<eos>')             
            tokens = [vocab[token] for token in item['tokens']] 
            data.append(tokens) 
    return data        

In [ ]:
train_data = prepare_tokenized(tokenized_dataset['train'], vocab)
test_data = prepare_tokenized(tokenized_dataset['test'], vocab)

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
class FlattenedSequenceDataset(Dataset):
    def __init__(self, sequences, window_size=5, num_batches=128):
        self.window_size = window_size
        self.num_batches = num_batches

        # Step 1: Flatten the sequences
        flat_sequence = [token for seq in sequences for token in seq]

        # Step 2: Determine the length of each subsequence
        total_length = len(flat_sequence) - window_size
        self.subseq_length = math.ceil(total_length / num_batches)

        # Create the input-target pairs
        self.data = []
        for i in range(0, total_length, self.subseq_length):
            subseq = flat_sequence[i:i+self.subseq_length + window_size]
            for j in range(len(subseq) - window_size):
                self.data.append((subseq[j:j+window_size], subseq[j+window_size]))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_seq, target = self.data[idx]
        return torch.tensor(input_seq), torch.tensor(target)

In [ ]:
batch_size = 128

In [ ]:
train_dataset = FlattenedSequenceDataset(train_data, 5, batch_size)
test_dataset = FlattenedSequenceDataset(test_data, 5, batch_size)

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=False, drop_last=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

In [ ]:
def train_rnn_language_model(model, train_loader, test_loader, loss_fn, optimizer, epochs, device, batch_size, clip=0.25):
    train_losses = []
    test_losses = []

    tqdm_epoch = trange(epochs, desc='Training')
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0
        total_train = 0
        hidden = model.init_hidden(batch_size)
        # Training loop
        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            # Zero the gradients
            model.zero_grad()
            hidden = hidden.detach()

            # Forward pass
            output, hidden = model(batch_X, hidden)
            
            # Reshape output for loss calculation
            output = output.view(-1, model.vocab_size)

            # Calculate the loss
            loss = loss_fn(output, batch_y)
            train_loss += loss.item() * batch_X.size(0)
            total_train += batch_y.size(0)

            # Backward pass and optimize
            loss.backward()
            #torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()

        train_loss /= total_train
        train_losses.append(train_loss)

        # Evaluation loop
        model.eval()
        test_loss = 0.0
        total_test = 0
        with torch.no_grad():
            hidden = model.init_hidden(batch_size)
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)
                
                # Forward pass
                output, hidden = model(batch_X, hidden)

                # Reshape output for loss calculation
                output = output.view(-1, model.vocab_size)

                # Calculate the loss
                loss = loss_fn(output, batch_y)
                test_loss += loss.item() * batch_X.size(0)
                total_test += batch_y.size(0)

        test_loss /= total_test
        test_losses.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} - Train PPL: {math.exp(train_loss):.4f}, Test PPL: {math.exp(test_loss):.4f}")

    return train_losses, test_losses

### Train a 2 layer GRU model

In [ ]:
embedding_dim = 1024
hidden_dim = 1024
n_layers = 2
dropout = 0.5
lr = 0.001
epochs = 20

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, n_layers, device, dropout=0.5):
        super(RNNLanguageModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.GRU(embedding_dim, hidden_dim, n_layers, dropout=dropout, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.device = device
        self.vocab_size = vocab_size
        self.embedding.weight = self.fc.weight

    def forward(self, x, hidden):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        last_output = output[:, -1, :]
        return self.fc(last_output), hidden

    def init_hidden(self, batch_size):
        return torch.zeros(self.rnn.num_layers, batch_size, self.rnn.hidden_size).to(self.device)

In [ ]:
rnn_model = RNNLanguageModel(vocab_len, embedding_dim, hidden_dim, n_layers, device, dropout).to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(rnn_model.parameters(), lr=lr)

In [ ]:
losses = train_rnn_language_model(rnn_model, train_dataloader, test_dataloader, 
                                  loss_fn, optimizer, epochs, device, batch_size)

In [ ]:
def predict_nextword(prompt, model, tokenizer, vocab, device):
    model.eval()
    tokens = tokenizer(prompt)
    indices = [vocab[t] for t in tokens]
    batch_size = 1
    hidden = model.init_hidden(batch_size)
    with torch.no_grad():
        src = torch.LongTensor([indices]).to(device)
        prediction, hidden = model(src, hidden)

        probs = torch.softmax(prediction, dim=-1)
        predicted_index = torch.argmax(probs, dim=1).item()

    itos = vocab.get_itos()
    token = itos[predicted_index]
    return token

In [ ]:
input_sequence = "The Turing test states that"

In [ ]:
predict_nextword(input_sequence, rnn_model, tokenizer, vocab, device)

In [ ]:
def generate(prompt, max_seq_len, temperature, model, tokenizer, vocab, device, seed=None):
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()
    tokens = tokenizer(prompt)
    indices = [vocab[t] for t in tokens]
    batch_size = 1
    hidden = model.init_hidden(batch_size)
    with torch.no_grad():
        for i in range(max_seq_len):
            src = torch.LongTensor([indices]).to(device)
            prediction, hidden = model(src, hidden)
            probs = torch.softmax(prediction/ temperature, dim=-1)  
            prediction = torch.multinomial(probs, num_samples=1).item()    
            
            while prediction == vocab['<unk>']:
                prediction = torch.multinomial(probs, num_samples=1).item()

            if prediction == vocab['<eos>']:
                break

            indices.append(prediction)

    itos = vocab.get_itos()
    tokens = [itos[i] for i in indices]
    return tokens

In [ ]:
prompt = 'Think about'
max_seq_len = 30
seed = 0

temperatures = [0.5, 0.7, 0.75, 0.8, 1.0]
for temperature in temperatures:
    generation = generate(prompt, max_seq_len, temperature, rnn_model, tokenizer, 
                          vocab, device, seed)
    print(str(temperature)+'\n'+' '.join(generation)+'\n')